# Transcription Factor Binding Site Classifier

This notebook trains a neural network to predict Rap1 binding sites (Step 3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from nn.nn import NeuralNetwork
from nn.io import read_text_file, read_fasta_file
from nn.preprocess import sample_seqs, one_hot_encode_seqs

## 1. Load Data

*Note: Ensure the data files are located in a `data/` directory or update the paths below.*

In [ ]:
# Replace these paths with the actual location of your data files
pos_seqs = read_text_file('data/rap1-lieb-positives.txt')
neg_seqs_raw = read_fasta_file('data/yeast-upstream-1k-negative.fa')

print(f"Number of positive sequences: {len(pos_seqs)}")
print(f"Length of positive sequences: {len(pos_seqs[0])}")

## 2. Process Negative Data

We need to process the long negative sequences into chunks of the same length as the positive sequences.

In [ ]:
motif_len = len(pos_seqs[0])
neg_seqs = []

# Subsample negative sequences to match the length of motif
# We utilize non-overlapping chunks from the negative fasta files
for seq in neg_seqs_raw:
    for i in range(0, len(seq) - motif_len + 1, motif_len):
        neg_seqs.append(seq[i:i+motif_len])

print(f"Number of generated negative sequences: {len(neg_seqs)}")

## 3. Balance Classes and Encode

In [ ]:
labels = [True] * len(pos_seqs) + [False] * len(neg_seqs)
all_seqs = pos_seqs + neg_seqs

# Use the sample_seqs function to balance the classes via upsampling
sampled_seqs, sampled_labels = sample_seqs(all_seqs, labels)

# One-hot encode the sequences
X = one_hot_encode_seqs(sampled_seqs)
y = np.array(sampled_labels).astype(int).reshape(-1, 1)

# Create Train/Val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Train Classifier

In [ ]:
input_dim = X.shape[1]

# Define Architecture
arch = [
    {'input_dim': input_dim, 'output_dim': 64, 'activation': 'relu'},
    {'input_dim': 64, 'output_dim': 1, 'activation': 'sigmoid'}
]

nn = NeuralNetwork(nn_arch=arch, lr=0.01, seed=42, batch_size=32, epochs=50, loss_function='binary_cross_entropy')

train_loss, val_loss = nn.fit(X_train, y_train, X_val, y_val)

## 5. Evaluation

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (BCE)')
plt.legend()
plt.title('Classifier Training Loss')
plt.show()

In [ ]:
preds = nn.predict(X_val)
predictions = (preds > 0.5).astype(int)
accuracy = np.mean(predictions == y_val)
print(f"Validation Accuracy: {accuracy:.4f}")

## 6. Explanation

**Sampling Scheme**: Since there are far more negative examples than positive ones, we used `sample_seqs` to upsample the positive class. This balances the dataset (1:1 ratio) ensuring the model does not bias heavily towards the majority class.

**Loss Function**: `binary_cross_entropy` is chosen as it is the standard loss function for binary classification problems (probability output).

**Hyperparameters**: 
- **Epochs (50)**: Sufficient for the model to learn the motif patterns.
- **Hidden Layer (64)**: Provides enough capacity to learn the sequence features without overfitting.
- **Learning Rate (0.01)**: Standard rate for stability.